In [1]:
# 1. 下载并安装 VS Code CLI 到稳定路径
from pathlib import Path
import os
import platform
import shutil
import stat
import subprocess
import tarfile
import tempfile
import urllib.request

FORCE_DOWNLOAD = False

home = Path.home()
dest = home / ".local/bin/code-tunnel-cli"
dest.parent.mkdir(parents=True, exist_ok=True)

def run_version(path):
    try:
        r = subprocess.run([str(path), "--version"], text=True, capture_output=True, timeout=20)
        return r.returncode, (r.stdout + r.stderr).strip()
    except Exception as e:
        return -1, repr(e)

if dest.exists() and not FORCE_DOWNLOAD:
    code, out = run_version(dest)
    print("existing:", dest)
    print(out)
else:
    machine = platform.machine().lower()
    if machine in {"x86_64", "amd64"}:
        os_name = "cli-alpine-x64"
    elif machine in {"aarch64", "arm64"}:
        os_name = "cli-alpine-arm64"
    else:
        raise RuntimeError(f"Unsupported architecture: {machine}")

    url = f"https://code.visualstudio.com/sha/download?build=stable&os={os_name}"
    print("downloading:", url)

    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        archive = td / "vscode-cli.tar.gz"
        urllib.request.urlretrieve(url, archive)

        extract_dir = td / "extract"
        extract_dir.mkdir()
        with tarfile.open(archive, "r:gz") as tf:
            tf.extractall(extract_dir)

        candidates = [p for p in extract_dir.rglob("code") if p.is_file()]
        if not candidates:
            raise RuntimeError("Downloaded archive did not contain a 'code' executable")

        shutil.copy2(candidates[0], dest)
        dest.chmod(dest.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

    code, out = run_version(dest)
    print("installed:", dest)
    print(out)

print("ready:", dest)

existing: /home/jovyan/.local/bin/code-tunnel-cli
code 1.127.0 (commit 4fe60c8b1cdac1c4c174f2fb180d0d758272d713)
ready: /home/jovyan/.local/bin/code-tunnel-cli


In [1]:
# 2. 后台启动 tunnel
# 如果第一次启动需要 GitHub device login，去日志里看登录网址和 code。
from pathlib import Path
import os
import signal
import subprocess
import time

TUNNEL_NAME = "curling-jupyter"
KILL_EXISTING_TUNNELS = True

code = Path.home() / ".local/bin/code-tunnel-cli"
log = Path.home() / ".vscode-tunnel.log"

if not code.exists():
    raise RuntimeError(f"VS Code CLI not found: {code}. Run the install cell first.")

def find_tunnel_processes():
    items = []
    for p in Path("/proc").iterdir():
        if not p.name.isdigit():
            continue
        try:
            cmd = (p / "cmdline").read_bytes().replace(b"\x00", b" ").decode("utf-8", "replace")
            if " tunnel " in f" {cmd} " and "code" in cmd:
                exe = os.readlink(p / "exe")
                items.append((int(p.name), exe, cmd))
        except Exception:
            pass
    return items

existing = find_tunnel_processes()
if existing:
    print("existing tunnel processes:")
    for pid, exe, cmd in existing:
        print(pid, exe, cmd[:300])

if existing and KILL_EXISTING_TUNNELS:
    for pid, exe, cmd in existing:
        try:
            os.kill(pid, signal.SIGTERM)
            print("SIGTERM", pid)
        except ProcessLookupError:
            pass
    time.sleep(3)
    for pid, exe, cmd in existing:
        if Path(f"/proc/{pid}").exists():
            try:
                os.kill(pid, signal.SIGKILL)
                print("SIGKILL", pid)
            except ProcessLookupError:
                pass

remaining = find_tunnel_processes()
if remaining:
    print("tunnel already running; not starting another one")
else:
    with log.open("ab", buffering=0) as f:
        p = subprocess.Popen(
            [str(code), "tunnel", "--accept-server-license-terms", "--name", TUNNEL_NAME],
            stdin=subprocess.PIPE,
            stdout=f,
            stderr=subprocess.STDOUT,
            start_new_session=True,
            close_fds=True,
        )
        # 兼容某些环境第一次启动时的选择提示；如果没有提示，这行也不会伤害。
        try:
            p.stdin.write(b"2\n")
            p.stdin.close()
        except Exception:
            pass
    print("started pid =", p.pid)

print("log =", log)
time.sleep(2)
print("current tunnel processes:")
for pid, exe, cmd in find_tunnel_processes():
    print("PID:", pid)
    print("EXE:", exe)
    print("CMD:", cmd[:500])

started pid = 64
log = /home/jovyan/.vscode-tunnel.log
current tunnel processes:
PID: 64
EXE: /home/jovyan/.local/bin/code-tunnel-cli
CMD: /home/jovyan/.local/bin/code-tunnel-cli tunnel --accept-server-license-terms --name curling-jupyter 


In [ ]:
# 3. 实时刷新显示 tunnel 日志最新 100 行
# 停止这个 cell 只会停止看日志，不会停止后台 tunnel。
from pathlib import Path
from IPython.display import clear_output
import time

log = Path.home() / ".vscode-tunnel.log"

print("watching", log)
print("按停止按钮可停止看日志，不会停 tunnel")
time.sleep(1)

while True:
    clear_output(wait=True)
    print("watching", log)
    print("按停止按钮可停止看日志，不会停 tunnel")
    print("=== latest 100 lines ===")

    if not log.exists():
        print("NO_LOG_YET: tunnel 可能还没写出日志。")
    else:
        lines = log.read_text(errors="replace").splitlines()
        print("size =", log.stat().st_size, "total_lines =", len(lines))
        print("\n".join(lines[-100:]))

    time.sleep(2)


watching /home/jovyan/.vscode-tunnel.log
按停止按钮可停止看日志，不会停 tunnel
=== latest 100 lines ===
size = 14175 total_lines = 196
[2026-07-06 06:03:19] info [rpc.8] Forwarding port 45977 (public=false)
[2026-07-06 06:03:19] info [tunnels::connections::relay_tunnel_host] Opened new client on channel 11
[2026-07-06 06:03:19] info [russh::server] wrote id
[2026-07-06 06:03:19] info [russh::server] read other id
[2026-07-06 06:03:19] info [russh::server] session is running
[2026-07-06 06:03:19] info [rpc.8] Forwarding port 46513 (public=false)
[2026-07-06 06:03:20] info [rpc.8] Forwarding port 46293 (public=false)
[2026-07-06 06:03:20] info [rpc.8] Forwarding port 41431 (public=false)
[2026-07-06 06:03:21] info [rpc.8] Forwarding port 41731 (public=false)
[2026-07-06 06:03:21] info [rpc.8] Forwarding port 56071 (public=false)
[2026-07-06 06:03:21] info [rpc.8] Forwarding port 53827 (public=false)
[2026-07-06 06:03:21] info [rpc.8] Forwarding port 52473 (public=false)
[2026-07-06 06:13:19] info [rpc.